In [1]:
import pandas as pd
import numpy as np
import os

# Display all columns
pd.set_option('display.max_columns', None)

print("Libraries loaded!")

Libraries loaded!


In [2]:
# Load the raw superstore dataset
df = pd.read_csv('../data/Sample - Superstore.csv', encoding='windows-1252')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
df.head()

Shape: (9994, 21)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Data types:
 Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Missing values:
 Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment 

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
# Fix date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Clean column names
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

print("Cleaned columns:", df.columns.tolist())
print("\nDate types fixed:")
print(df[['order_date', 'ship_date']].dtypes)

Cleaned columns: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub-category', 'product_name', 'sales', 'quantity', 'discount', 'profit']

Date types fixed:
order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object


In [4]:
# ── TABLE 1: dim_customers ───────────────────────────────────────
dim_customers = df[['customer_id', 'customer_name', 
                    'segment']].drop_duplicates().reset_index(drop=True)
print(f"dim_customers: {dim_customers.shape}")

# ── TABLE 2: dim_products ────────────────────────────────────────
dim_products = df[['product_id', 'product_name', 
                   'category', 'sub-category']].drop_duplicates().reset_index(drop=True)
dim_products.columns = ['product_id', 'product_name', 'category', 'sub_category']
print(f"dim_products: {dim_products.shape}")

# ── TABLE 3: dim_location ────────────────────────────────────────
dim_location = df[['postal_code', 'city', 'state', 
                   'country', 'region']].drop_duplicates().reset_index(drop=True)
print(f"dim_location: {dim_location.shape}")

# ── TABLE 4: dim_shipping ────────────────────────────────────────
dim_shipping = df[['order_id', 'ship_date', 
                   'ship_mode']].drop_duplicates().reset_index(drop=True)
print(f"dim_shipping: {dim_shipping.shape}")

# ── TABLE 5: fact_orders (main fact table) ───────────────────────
fact_orders = df[['row_id', 'order_id', 'order_date', 
                  'customer_id', 'product_id', 'postal_code',
                  'sales', 'quantity', 'discount', 'profit']].copy()
print(f"fact_orders: {fact_orders.shape}")

dim_customers: (793, 3)
dim_products: (1894, 4)
dim_location: (632, 5)
dim_shipping: (5009, 3)
fact_orders: (9994, 10)


In [5]:
# Add useful calculated columns to fact table
fact_orders['profit_margin']  = round(fact_orders['profit'] / fact_orders['sales'] * 100, 2)
fact_orders['shipping_days']  = (df['ship_date'] - df['order_date']).dt.days
fact_orders['order_year']     = fact_orders['order_date'].dt.year
fact_orders['order_month']    = fact_orders['order_date'].dt.month
fact_orders['order_quarter']  = fact_orders['order_date'].dt.quarter

# Flag negative profit orders
fact_orders['is_loss'] = fact_orders['profit'] < 0

print("New columns added:")
print(fact_orders[['profit_margin', 'shipping_days', 
                   'order_year', 'order_quarter', 'is_loss']].head())
print(f"\nLoss-making orders: {fact_orders['is_loss'].sum():,}")
print(f"Average profit margin: {fact_orders['profit_margin'].mean():.1f}%")

New columns added:
   profit_margin  shipping_days  order_year  order_quarter  is_loss
0          16.00              3        2016              4    False
1          30.00              3        2016              4    False
2          47.00              4        2016              2    False
3         -40.00              7        2015              4     True
4          11.25              7        2015              4    False

Loss-making orders: 1,871
Average profit margin: 12.0%


In [6]:
# Save all dimension and fact tables
dim_customers.to_csv('../data/dim_customers.csv', index=False)
dim_products.to_csv('../data/dim_products.csv', index=False)
dim_location.to_csv('../data/dim_location.csv', index=False)
dim_shipping.to_csv('../data/dim_shipping.csv', index=False)
fact_orders.to_csv('../data/fact_orders.csv', index=False)

print("All 5 tables saved!")
print("\nFiles in data folder:")
import os
for f in os.listdir('../data'):
    size = os.path.getsize(f'../data/{f}') / 1024
    print(f"  {f:<30} {size:.1f} KB")

All 5 tables saved!

Files in data folder:
  dim_customers.csv              26.2 KB
  dim_location.csv               28.5 KB
  dim_products.csv               143.5 KB
  dim_shipping.csv               199.7 KB
  fact_orders.csv                1029.3 KB
  Sample - Superstore.csv        2234.2 KB


## Data Cleaning & Table Design Summary

### Raw Data
- Source: Sample Superstore (Kaggle)
- Shape: 9,994 rows × 21 columns
- Missing values: None — clean dataset

### Star Schema Design
| Table | Rows | Description |
|-------|------|-------------|
| fact_orders | 9,994 | Central fact table — all transactions |
| dim_customers | 793 | Unique customers with segments |
| dim_products | 1,894 | Products with categories |
| dim_location | 632 | Geographic locations |
| dim_shipping | 5,009 | Shipment details |

### Feature Engineering
- `profit_margin` — profit as % of sales
- `shipping_days` — days from order to ship
- `order_year/month/quarter` — date breakdowns
- `is_loss` — flags negative profit orders

### Key Findings
- 1,871 orders (18.7%) are loss-making
- Average profit margin: 12.0%
- Next step: SQL analysis